# Evaluar para generalizar

Métricas, particiones y validación cruzada

## El problema de la generalización

Los capítulos anteriores resolvieron el problema de **ajuste**. Fijada una clase de hipótesis, construimos el riesgo empírico sobre la muestra disponible y lo minimizamos para obtener $\hat{\boldsymbol{w}}$. El resultado es el modelo concreto $f_{\hat{\boldsymbol{w}}}$: una ecuación cuyos parámetros ya están fijados.

No obstante, esto no resuelve el problema de **generalización**. Al encontrar los parámetros $\hat{\boldsymbol{w}}$ minimizando sobre la muestra de datos disponible, el procedimiento aprovecha tanto la señal como las fluctuaciones accidentales (ruido) de esa muestra. Por ello, el riesgo empírico del modelo ajustado, evaluado sobre las muestras utilizadas para el entrenamiento, es optimista. El interpolador de **?@def-sobreajuste** era el caso extremo.

Dado un modelo ajustado $\hat{\boldsymbol{w}}$, la medida más razonable de rendimiento no es el riesgo de entrenamiento, sino el **riesgo verdadero** de **?@def-riesgo-verdadero**,

$$
R(\hat{\boldsymbol{w}})
=\mathbb{E}_{P^\star}\!\left[ \ell\bigl(Y,f_{\hat{\boldsymbol{w}}}(X)\bigr) \right],
$$

es decir, la pérdida media que el modelo sufriría sobre nuevas extracciones de la distribución de los datos $P^\star$. El problema es que en la práctica, no conocemos $P^\star$, tan solo tenemos acceso a una muestra finita de esta distribución.

## Las dos tareas

En este capítulo abordaremos dos de las tareas fundamentales del aprendizaje automático. Explicaremos cómo afrontar cada una de ellas aprovechando al máximo los datos disponibles.

**Tarea 1: Estimar el riesgo verdadero de un modelo ya ajustado.** Una vez fijado un modelo $f_{\hat{\boldsymbol{w}}}$, nuestro objetivo es estimar su riesgo verdadero $R(\hat{\boldsymbol{w}})$ y cuantificar la precisión de dicha estimación. Este es el valor de rendimiento esperado que debe acompañar a cualquier modelo en su entrega final.

**Tarea 2: Elegir el mejor procedimiento entre $M$ alternativas.** En la práctica, rara vez contamos con un único modelo $f_{\hat{\boldsymbol{w}}}$. Normalmente necesitamos explorar varias alternativas antes de decidir el modelo final. Por ejemplo, tenemos que tomar decisiones como si merece la pena introducir una variable predictora extra. Al hacer esto, no estamos comparando modelos terminados, sino procedimientos de aprendizaje. Un procedimiento es, esencialmente, una “receta” que recibe datos de entrenamiento y produce un modelo; por tanto, evaluar qué procedimiento elegir requiere un enfoque analítico distinto al de evaluar un modelo cuyos parámetros ya están congelados.

Antes de resolver ambas tareas, estableceremos una referencia de rendimiento mínima: el modelo nulo. Dado que una cifra de riesgo aislada carece de escala interpretativa, necesitamos fijar este punto de partida elemental para saber si nuestros procedimientos aprenden algo útil.

### Empezar por una referencia sencilla

La elección natural contra la que comparar el riesgo de un modelo dado es evaluar el riesgo de un modelo que ignora por completo las variables predictoras. Consideremos la familia de funciones constantes, $\{f_c(\mathbf{x})=c:c\in\mathbb{R}\}$, y el procedimiento que elige el valor de $c$ minimizando el riesgo empírico.

<span class="theorem-title">**Definición 1 (Modelo nulo)**</span> El modelo nulo $\bar{f}$ es el modelo que predice exactamente el mismo valor para todas las observaciones, sin utilizar $\mathbf{x}$.

Queda por decidir qué valor. Este valor será el que minimice el riesgo empírico sobre el conjunto de entrenamiento, y dependerá de la función de coste que elijamos.

<span class="theorem-title">**Lema 1 (Bajo coste cuadrático, la mejor constante es la media)**</span> Sea $\hat{R}(c)=\frac{1}{n}\sum_{i=1}^{n}(y_i-c)^2$. Entonces

$$
\mathop{\mathrm{arg\,min}}_{c\in\mathbb{R}}\hat{R}(c)=\bar{y},
\qquad
\hat{R}(\bar{y})=\frac{1}{n}\sum_{i=1}^{n}(y_i-\bar{y})^2.
$$

<span class="proof-title">*Demostración*. </span>La función $\hat{R}(c)$ es derivable en $c$, y

$$
\hat{R}'(c)=-\frac{2}{n}\sum_{i=1}^{n}(y_i-c)
=-2\bigl(\bar{y}-c\bigr).
$$

Se anula solo en $c=\bar{y}$. Además $\hat{R}''(c)=2>0$, así que ese punto crítico es el mínimo, y es el único. Sustituyendo $c=\bar{y}$ se obtiene el valor del enunciado.

Durante este capítulo, ilustraremos los conceptos que vayan apareciendo usando los datos de próstata del capítulo anterior. El fichero conserva la partición de *The Elements of Statistical Learning*: 67 observaciones de entrenamiento y 30 observaciones que permanecerán aisladas hasta que formalicemos su uso.

In [ ]:
import numpy as np
import pandas as pd
import torch
from matplotlib import pyplot as plt

torch.set_default_dtype(torch.float64)
torch.manual_seed(42)

prostata = pd.read_csv("../datos/prostate.data")
predictoras = [
    "lcavol", "lweight", "age", "lbph",
    "svi", "lcp", "gleason", "pgg45",
]

X = torch.cat([
    torch.ones(len(prostata), 1),
    torch.tensor(prostata[predictoras].values),
], dim=1)
y = torch.tensor(prostata["lpsa"].values)

es_train = torch.tensor(
    (prostata["train"].str.strip() == "T").values
)
X_train, y_train = X[es_train], y[es_train]
X_test, y_test = X[~es_train], y[~es_train]

AZUL, GRIS, NARANJA, TINTA = "#151f6c", "#8b8e95", "#ff5700", "#1b1d21"
kw_puntos = dict(facecolors="none", s=40, alpha=0.65)

El siguiente código separa con claridad el **procedimiento de ajuste**, `ajusta_lineal`, del **modelo** que devuelve, `ModeloLineal`, cuyos coeficientes ya están fijados.

In [ ]:
# TODO: completar en clase

Con esto ya podemos fijar la referencia nula y compararla, más adelante, con el modelo de ocho variables.

In [ ]:
# TODO: completar en clase

El modelo nulo predice la constante $2.4523$ y tiene riesgo $1.4370$ en train.

<span class="theorem-title">**Ejercicio 1 (La mejor constante con otra pérdida)**</span> El resultado de <a href="#lem-media-optima" class="quarto-xref">Lema 1</a> depende de la pérdida cuadrática.

1.  Escribe el riesgo empírico de una constante $c$ con la pérdida absoluta, $\frac{1}{n}\sum_i\left\lvert y_i-c \right\rvert$, y explica por qué no se puede derivar en todo punto.
2.  Argumenta, moviendo $c$ un poco a la derecha y contando cuántas observaciones quedan a cada lado, que el mínimo se alcanza en la mediana de la muestra.
3.  Calcula con `torch` la media y la mediana de `y_train` y comprueba cuál de las dos da menor error absoluto medio.

## Tarea 1: estimar el riesgo de un modelo ya ajustado

Comenzamos explicando cómo resolver la primera de las tareas anunciadas. Imaginemos que un modelo ajustado con una base de datos $\mathcal{D}_{\mathrm{train}}$ produce parámetros $\hat{\boldsymbol{w}}$. Como hemos discutido, el riesgo verdadero de este modelo es:

$$
R(\hat{\boldsymbol{w}})
=\mathbb{E}_{P^\star}\!\left[ \ell\bigl(Y,f_{\hat{\boldsymbol{w}}}(X)\bigr) \right] .
$$

La forma ingenua de estimar este riesgo consistiría en promediar las pérdidas sobre el conjunto de datos $\mathcal{D}_{\mathrm{train}}$. Es decir, calculando:

$$
\hat{R}_{\mathcal{D}_{\mathrm{train}}}(\hat{\boldsymbol{w}})
=\frac{1}{n_{\mathrm{train}}}\sum_{i\in\mathcal{D}_{\mathrm{train}}}
  \ell\bigl(y_i,f_{\hat{\boldsymbol{w}}}(\mathbf{x}_i)\bigr) .
$$

El problema es que $\hat{\boldsymbol{w}}$ se obtuvo minimizando precisamente esa media. Veremos que esta medida de riesgo es una estimación optimista del riesgo verdadero.

<span class="theorem-title">**Proposición 1 (El riesgo en train es optimista en media)**</span> Sean las observaciones de $\mathcal{D}_{\mathrm{train}}$ extracciones independientes de $P^\star$. Sea $\hat{\boldsymbol{w}}$ el minimizador del riesgo empírico sobre $\mathcal{D}_{\mathrm{train}}$ y sea $\boldsymbol{w}^\star$ un minimizador del riesgo verdadero $R(\boldsymbol{w})=\mathbb{E}_{P^\star}\!\left[ \ell(Y,f_{\boldsymbol{w}}(X)) \right]$ de **?@eq-riesgo-verdadero**. Con las dos esperanzas tomadas sobre muestras de entrenamiento de tamaño $n_{\mathrm{train}}$,

$$
\mathbb{E}_{\mathcal{D}_{\mathrm{train}}}\!\left[ \hat{R}_{\mathcal{D}_{\mathrm{train}}}(\hat{\boldsymbol{w}}) \right]\;\leq\;R(\boldsymbol{w}^\star)\;\leq\;
\mathbb{E}_{\mathcal{D}_{\mathrm{train}}}\!\left[ R(\hat{\boldsymbol{w}}) \right] .
$$

<span class="proof-title">*Demostración*. </span>Para la primera desigualdad, $\hat{\boldsymbol{w}}$ minimiza $\hat{R}_{\mathcal{D}_{\mathrm{train}}}$, luego $\hat{R}_{\mathcal{D}_{\mathrm{train}}}(\hat{\boldsymbol{w}})\leq\hat{R}_{\mathcal{D}_{\mathrm{train}}}(\boldsymbol{w}^\star)$ sea cual sea la muestra que salga. Tomando esperanzas, y usando que $\boldsymbol{w}^\star$ está fijado sin mirar la muestra, de modo que por **?@lem-esperanza-funcion** la esperanza de cada pérdida vale $R(\boldsymbol{w}^\star)$,

$$
\mathbb{E}_{\mathcal{D}_{\mathrm{train}}}\!\left[ \hat{R}_{\mathcal{D}_{\mathrm{train}}}(\hat{\boldsymbol{w}}) \right]
\leq\mathbb{E}_{\mathcal{D}_{\mathrm{train}}}\!\left[ \hat{R}_{\mathcal{D}_{\mathrm{train}}}(\boldsymbol{w}^\star) \right]=R(\boldsymbol{w}^\star).
$$

Para la segunda, $R(\boldsymbol{w}^\star)\leq R(\boldsymbol{w})$ para todo $\boldsymbol{w}$ por definición de $\boldsymbol{w}^\star$, en particular para $\hat{\boldsymbol{w}}$, y las desigualdades se conservan al tomar esperanzas.

Esto nos dice que el riesgo $\hat{R}_{\mathcal{D}_{\mathrm{train}}}(\hat{\boldsymbol{w}})$ es, en promedio, menor o igual que el riesgo verdadero del modelo ajustado, $R(\hat{\boldsymbol{w}})$. Es consecuencia de evaluar $\hat{\boldsymbol{w}}$ sobre las mismas observaciones con las que se calculó, y el resultado siguiente dice qué pasa cuando eso no ocurre.

<span class="theorem-title">**Teorema 1 (El error de test es insesgado para un modelo fijo)**</span> Sea $\hat{\boldsymbol{w}}$ un vector de parámetros **fijado sin usar la muestra $\mathcal{D}_{\mathrm{test}}$**, y sean las observaciones de $\mathcal{D}_{\mathrm{test}}$ extracciones independientes de $P^\star$. Entonces

$$
\mathbb{E}_{\mathcal{D}_{\mathrm{test}}}\!\left[ \hat{R}_{\mathcal{D}_{\mathrm{test}}}(\hat{\boldsymbol{w}}) \right]=R(\hat{\boldsymbol{w}}),
$$

donde la esperanza se toma sobre las muestras posibles de ese tamaño.

<span class="proof-title">*Demostración*. </span>Por **?@def-riesgo-empirico**,

$$
\hat{R}_{\mathcal{D}_{\mathrm{test}}}(\boldsymbol{w})
=\frac{1}{n_{\mathrm{test}}}\sum_{i\in\mathcal{D}_{\mathrm{test}}}\ell\bigl(y_i,f_{\boldsymbol{w}}(\mathbf{x}_i)\bigr).
$$

Fijado $\boldsymbol{w}$, la función $g(\mathbf{x},y)=\ell\bigl(y,f_{\boldsymbol{w}}(\mathbf{x})\bigr)$ está elegida sin mirar $\mathcal{D}_{\mathrm{test}}$, y cada observación $(\mathbf{x}_i,y_i)$ de ese conjunto es una extracción de $P^\star$. Por **?@lem-esperanza-funcion**, la esperanza de cada sumando es la integral de $g$ contra $P^\star$, es decir $\mathbb{E}_{P^\star}\!\left[ \ell\bigl(Y,f_{\boldsymbol{w}}(X)\bigr) \right]=R(\boldsymbol{w})$, que es la definición de **?@eq-riesgo-verdadero** y es la misma para todos los sumandos. La esperanza es lineal, así que la esperanza de la media es la media de las esperanzas:

$$
\mathbb{E}_{\mathcal{D}_{\mathrm{test}}}\!\left[ \hat{R}_{\mathcal{D}_{\mathrm{test}}}(\boldsymbol{w}) \right]
=\frac{1}{n_{\mathrm{test}}}\sum_{i\in\mathcal{D}_{\mathrm{test}}}R(\boldsymbol{w})
=R(\boldsymbol{w}).
$$

En suma, si conociéramos $P^\star$, dado un modelo ajustado con parámetros $\hat{\boldsymbol{w}}$, podríamos calcular $R(\hat{\boldsymbol{w}})$ integrando directamente. Como $P^\star$ es desconocida, la única aproximación disponible es otra media empírica; <a href="#prp-train-optimista" class="quarto-xref">Proposición 1</a> obliga a calcularla sobre observaciones que no hayan intervenido en el ajuste. Esto motiva la primera partición.

<span class="theorem-title">**Definición 2 (Entrenamiento y test)**</span> Para poder ajustar un modelo predictivo y estimar su riesgo de generalización, la muestra $\mathcal{D}$ debe repartirse **al azar** en dos partes disjuntas:

- $\mathcal{D}_{\mathrm{train}}$, de tamaño $n_{\mathrm{train}}$, con la que el procedimiento fija los parámetros del modelo;
- $\mathcal{D}_{\mathrm{test}}$, de tamaño $n_{\mathrm{test}}$, con la que se mide el riesgo de ese modelo, sin haberla consultado antes.

Escribimos $\hat{R}_{\mathcal{D}_{\mathrm{test}}}(\boldsymbol{w})$ para el riesgo empírico calculado sobre $\mathcal{D}_{\mathrm{test}}$, y análogamente para $\mathcal{D}_{\mathrm{train}}$.

El reparto aleatorio hace que ambos subconjuntos representen la misma $P^\star$.

Por <a href="#thm-test-insesgado" class="quarto-xref">Teorema 1</a>, condicionalmente a $\mathcal{D}_{\mathrm{train}}$, $\hat{R}_{\mathcal{D}_{\mathrm{test}}}(\hat{\boldsymbol{w}})$ es una estimación insesgada de $R(\hat{\boldsymbol{w}})$ siempre que $\mathcal{D}_{\mathrm{test}}$ permanezca ajeno a toda decisión tomada para calcular $\hat{\boldsymbol{w}}$. No obstante, no olvidemos que $\hat{R}_{\mathcal{D}_{\mathrm{test}}}(\hat{\boldsymbol{w}})$ es tan solo una estimación. La precisión de la misma, dependerá del tamaño de $\mathcal{D}_{\mathrm{test}}$.

### Con qué precisión se mide

La estimación de test es una media de $n_{\mathrm{test}}$ números, y como cualquier media tiene una varianza que depende del tamaño de la muestra con la que se calcula.

<span class="theorem-title">**Proposición 2 (Precisión de la estimación de test)**</span> En las condiciones de <a href="#thm-test-insesgado" class="quarto-xref">Teorema 1</a>,

$$
\mathrm{Var}_{\mathcal{D}_{\mathrm{test}}}\!\left( \hat{R}_{\mathcal{D}_{\mathrm{test}}}(\boldsymbol{w}) \right)
=\frac{1}{n_{\mathrm{test}}}\mathrm{Var}_{P^\star}\!\left( \ell\bigl(Y,f_{\boldsymbol{w}}(X)\bigr) \right) .
$$

El subíndice importa: a la izquierda la varianza es sobre las muestras de test de tamaño $n_{\mathrm{test}}$, y a la derecha sobre una sola observación nueva.

<span class="proof-title">*Demostración*. </span>Las $n_{\mathrm{test}}$ pérdidas son independientes y tienen todas la misma varianza, que llamamos $v$. La varianza de su suma es $n_{\mathrm{test}}\,v$, y multiplicar por $1/n_{\mathrm{test}}$ multiplica la varianza por $1/n_{\mathrm{test}}^2$, de modo que queda $v/n_{\mathrm{test}}$.

El error típico de la estimación es entonces $\sqrt{v/n_{\mathrm{test}}}$, que decrece a un ritmo de $1/\sqrt{n_{\mathrm{test}}}$: para reducir la incertidumbre a la mitad, es necesario cuadruplicar el tamaño del conjunto de test. En la práctica, no obstante, la varianza $v$ de las pérdidas es desconocida. El procedimiento riguroso consiste en aproximarla utilizando la varianza muestral (insesgada) de las pérdidas individuales evaluadas sobre $\mathcal{D}_{\mathrm{test}}$, que denotaremos como $s^2$. Por ello, el estándar para informar sobre el rendimiento de generalización de un modelo consiste en reportar su estimación puntual $\hat{R}_{\mathcal{D}_{\mathrm{test}}}(\boldsymbol{w})$ junto con un margen de $\pm$ un error típico estimado:

$$
\widehat{\text{EE}} = \frac{s}{\sqrt{n_{\mathrm{test}}}} = \sqrt{ \frac{1}{n_{\mathrm{test}}(n_{\mathrm{test}}- 1)} \sum_{i \in \mathcal{D}_{\mathrm{test}}} \left( \ell\bigl(y_i, f_{\boldsymbol{w}}(\mathbf{x}_i)\bigr) - \hat{R}_{\mathcal{D}_{\mathrm{test}}}(\boldsymbol{w}) \right)^2 }
$$

Esto ha de interpretarse de la siguiente manera: al calcular el riesgo empírico como una media de variables aleatorias independientes e idénticamente distribuidas, el Teorema Central del Límite garantiza que, para un $n_{\mathrm{test}}$ suficientemente grande, la distribución de nuestra estimación converge a una normal. Por tanto, el intervalo construido al sumar y restar un error típico al riesgo estimado capturará el riesgo verdadero $R(\boldsymbol{w})$ en aproximadamente un 68 % de las realizaciones de $\mathcal{D}_{\mathrm{test}}$.

<span class="theorem-title">**Ejercicio 2 (Cuánto test hace falta)**</span> Supón pérdidas cuadráticas con ruido gaussiano de nivel $\sigma$, para las que se puede comprobar que $\mathrm{Var}\!\left( \ell \right)=2\sigma^4$.

1.  Usando <a href="#prp-precision-test" class="quarto-xref">Proposición 2</a>, escribe el error típico de $\hat{R}_{\mathcal{D}_{\mathrm{test}}}$ en función de $\sigma$ y de $n_{\mathrm{test}}$.
2.  Divide ese error típico por el propio riesgo $\sigma^2$ y comprueba que el error **relativo** es $\sqrt{2/n_{\mathrm{test}}}$, que no depende de $\sigma$.
3.  Calcula cuántas observaciones de test hacen falta para un error relativo del 20 % y para uno del 10 %.
4.  Compara el resultado con las 30 observaciones de test de los datos de próstata.

## Tarea 2: elegir el mejor procedimiento entre $M$ alternativas

Con lo expuesto en la sección anterior, ya sabemos que necesitamos reservar una parte de la muestra de datos ($\mathcal{D}_{\mathrm{test}}$) exclusivamente para evaluar un modelo y reportar una estimación honesta de su error de generalización.

En la práctica, no obstante, rara vez ejecutamos una única estrategia de ajuste. Usualmente necesitamos explorar y elegir entre $M$ alternativas. En los datos de próstata, por ejemplo, podemos probar a ajustar modelos usando solo la variable más correlacionada con la respuesta, las dos más correlacionadas, y así sucesivamente hasta considerar las ocho. La pregunta natural es cuál de estas alternativas es mejor.

Si conociéramos la distribución poblacional $P^\star$, la respuesta sería directa. Aplicaríamos las $M$ alternativas a nuestro conjunto de entrenamiento $\mathcal{D}_{\mathrm{train}}$ para obtener $M$ modelos concretos. Luego, integraríamos sobre $P^\star$ para calcular el riesgo verdadero exacto de cada uno de esos modelos, y simplemente entregaríamos el que tuviera el valor más bajo.

Sin embargo, en la realidad no conocemos $P^\star$, así que no podemos calcular el riesgo exacto de los $M$ modelos. Como disponemos de un conjunto $\mathcal{D}_{\mathrm{test}}$ reservado, y sabemos que el error medido en él es insesgado (por <a href="#thm-test-insesgado" class="quarto-xref">Teorema 1</a>), una estrategia aparentemente razonable sería la siguiente:

1.  Ajustar los $M$ modelos usando todos los datos de $\mathcal{D}_{\mathrm{train}}$.

2.  Estimar el riesgo verdadero de esos $M$ modelos evaluándolos en $\mathcal{D}_{\mathrm{test}}$.

3.  Seleccionar el modelo que tenga menor error en test.

Esta estrategia falla, por la razón siguiente.

<span class="theorem-title">**Proposición 3 (El mínimo de varias estimaciones insesgadas es optimista)**</span> Sean $Z_1,\ldots,Z_M$ estimaciones y $q_1,\ldots,q_M$ las cantidades que estiman, cada una sin sesgo: $\mathbb{E}\!\left[ Z_k \right]=q_k$ para todo $k$. Entonces

$$
\mathbb{E}\!\left[ \min_{k}Z_k \right]\;\leq\;\min_{k}q_k .
$$

<span class="proof-title">*Demostración*. </span>Para cada $j$ se cumple $\min_k Z_k\leq Z_j$, porque el mínimo de un conjunto es menor o igual que cualquiera de sus elementos. Tomando esperanzas, que conservan las desigualdades, $\mathbb{E}\!\left[ \min_k Z_k \right]\leq\mathbb{E}\!\left[ Z_j \right]=q_j$. Como esto vale para todo $j$, vale también para el $j$ que hace $q_j$ más pequeño.

Para entender las consecuencias de esta proposición, apliquémosla a nuestro caso. Fijado $\mathcal{D}_{\mathrm{train}}$, tenemos $M$ modelos concretos. Los $Z_k$ representan sus riesgos estimados en el test, y los $q_k$ sus verdaderos riesgos de generalización. La proposición demuestra que $\mathbb{E}_{\mathcal{D}_{\mathrm{test}}}\!\left[ \min_k Z_k \right]\leq\min_k q_k$. Es decir, la estimación que obtenemos al quedarnos con el mínimo es, en promedio, excesivamente optimista.

El modelo que menor riesgo de test consigue puede haber obtenido un error bajo por 1) tratarse de un modelo genuinamente bueno (un $q_k$ bajo) o 2) ha tenido suerte con las observaciones específicas que componen esa muestra de test (una fluctuación favorable en $Z_k$). Cuantos más candidatos evaluemos, más probable es que alguno parezca excelente por puro accidente.

Como consecuencia directa de buscar este mínimo, el riesgo medido para el modelo ganador está siempre sesgado a la baja. Lo hemos “coronado” precisamente porque ha sido capaz de explotar las peculiaridades de esa muestra concreta para presentar la mejor métrica posible.

Esta es la razón fundamental por la que evaluar para elegir “quema” la muestra utilizada. En el instante en que $\mathcal{D}_{\mathrm{test}}$ interviene para decidir qué modelo seleccionar, la estimación de riesgo resultante deja de ser independiente del modelo elegido, rompiendo la garantía de <a href="#thm-test-insesgado" class="quarto-xref">Teorema 1</a>. Si la estimación que acompaña al modelo final debe ser honesta, se deduce una regla estricta de partición: el conjunto de datos empleado para elegir el mejor procedimiento debe ser estrictamente distinto al conjunto empleado para evaluar su rendimiento final.

El siguiente experimento estudia este fenómeno. Se comparan varios procedimientos intercambiables: todos producen una recta usando, además, una variable de ruido distinta. Se elige con 40 observaciones apartadas, que hacen el papel del conjunto sobre el que la estrategia ingenua elige e informa, y se mide el modelo seleccionado sobre otras 4000, que aproximan su riesgo verdadero.

In [ ]:
# TODO: completar en clase

In [ ]:
torch.manual_seed(0)
repeticiones = 2000
resultados = []
for M in (1, 2, 5, 10, 20, 50):
    acumulado = torch.zeros(2)
    for _ in range(repeticiones):
        acumulado += torch.tensor([float(v)
                                   for v in experimento_seleccion(M)])
    resultados.append((M, *(acumulado / repeticiones).tolist()))

print(" M   validación del elegido   test del elegido")
for M, v, t in resultados:
    print(f"{M:3d}         {v:.4f}                {t:.4f}")

La segunda columna, el riesgo de test del modelo elegido, apenas se mueve. La primera columna, en cambio, baja en los seis casos, de $1.084$ con un candidato a $0.956$ con cincuenta. Las dos miden el mismo modelo, y la única diferencia entre ellas es que la primera se calcula sobre el conjunto que se usó para elegirlo.

Con un solo candidato las dos columnas prácticamente coinciden, $1.084$ frente a $1.078$, que es lo que dice <a href="#thm-test-insesgado" class="quarto-xref">Teorema 1</a> aplicado a ese conjunto: cuando no se elige, la estimación es honesta. Con cincuenta candidatos la separación es de $0.14$, un 13 % del valor.

Figura 1: Lo que le pasa a la estimación cuando se prueban más candidatos. La línea negra aproxima el riesgo verdadero del modelo elegido y <span class="nuevo">apenas se mueve</span> porque los candidatos son intercambiables por construcción. La naranja es el número que uno estaría tentado de informar y se separa cada vez más de la negra. Con veinte candidatos cruza el suelo de ruido $\sigma^2=1$, y a partir de ahí informa de un riesgo menor que el que cualquier modelo de este experimento puede alcanzar.

In [ ]:
Ms = [r[0] for r in resultados]

fig, ax = plt.subplots(figsize=(6.4, 4.0))
ax.plot(Ms, [r[1] for r in resultados], marker="o", ms=4, color=NARANJA,
        label="validación del modelo elegido")
ax.plot(Ms, [r[2] for r in resultados], marker="s", ms=4, color=TINTA,
        label="test del modelo elegido")
ax.axhline(1.0, color=GRIS, lw=1, linestyle=":",
           label=r"suelo de ruido $\sigma^2$")
ax.set_xscale("log")
ax.set_xticks(Ms)
ax.set_xticklabels([str(m) for m in Ms])
ax.set(xlabel="número de candidatos probados",
       ylabel="riesgo cuadrático medio")
ax.legend(fontsize=8.5)
plt.tight_layout()

### Validación

La conclusión de lo anterior es inequívoca. $\mathcal{D}_{\mathrm{test}}$ solo conserva la garantía de <a href="#thm-test-insesgado" class="quarto-xref">Teorema 1</a> si no participa en la Tarea 2. Debe aislarse desde el principio y abrirse una sola vez, cuando el procedimiento ya está elegido y el modelo final ya está fijado.

Por tanto, toda la selección debe ocurrir usando exclusivamente $\mathcal{D}_{\mathrm{train}}$. Sin embargo, no podemos usar la totalidad de $\mathcal{D}_{\mathrm{train}}$ para ajustar los $M$ candidatos y a la vez para elegir al ganador. Como demostró <a href="#prp-train-optimista" class="quarto-xref">Proposición 1</a>, evaluar un modelo sobre sus propios datos de ajuste produce un riesgo falsamente optimista. Peor aún, este sesgo beneficia sistemáticamente a los modelos más complejos, que parecerían siempre los ganadores por puro sobreajuste, arruinando la comparación. Por tanto, para evaluar las alternativas de forma justa, estamos obligados a fraccionar $\mathcal{D}_{\mathrm{train}}$: una parte para ajustar los candidatos y otra independiente para seleccionarlos.

La forma más inmediata de ejecutar este fraccionamiento es la **validación simple**. Apartamos temporalmente un subconjunto $\mathcal{D}_{\mathrm{val}}\subset \mathcal{D}_{\mathrm{train}}$ de tamaño $n_{\mathrm{val}}$. El resto de los datos, que llamaremos $\mathcal{D}_{\mathrm{ajuste}}$ y tiene $n_{\mathrm{train}}- n_{\mathrm{val}}$ observaciones, se utiliza para ajustar las $M$ alternativas. Esto produce $M$ modelos provisionales que evaluamos y comparamos sobre $\mathcal{D}_{\mathrm{val}}$. A nivel práctico, esto implica la siguiente partición de los datos.

<span class="theorem-title">**Definición 3 (Entrenamiento, validación y test)**</span> Primero se separa $\mathcal{D}_{\mathrm{test}}$ al azar y se denomina $\mathcal{D}_{\mathrm{train}}$ al resto de los datos. Una validación simple parte a su vez $\mathcal{D}_{\mathrm{train}}=\mathcal{D}_{\mathrm{ajuste}}\cup\mathcal{D}_{\mathrm{val}}$, con lo que quedan tres subconjuntos disjuntos y tres papeles distintos:

- $\mathcal{D}_{\mathrm{ajuste}}$, con $n_{\mathrm{train}}-n_{\mathrm{val}}$ observaciones, fija los parámetros de cada modelo provisional, y por eso el riesgo medido sobre él es optimista según <a href="#prp-train-optimista" class="quarto-xref">Proposición 1</a>;
- $\mathcal{D}_{\mathrm{val}}$, con $n_{\mathrm{val}}$ observaciones, compara los procedimientos candidatos y queda consumido por esa elección, de modo que el riesgo del ganador medido en él es optimista según <a href="#prp-minimo-sesgado" class="quarto-xref">Proposición 3</a>;
- $\mathcal{D}_{\mathrm{test}}$, con $n_{\mathrm{test}}$ observaciones, mide una sola vez el modelo final y no interviene en ninguna decisión, que es lo que hace insesgada esa medición por <a href="#thm-test-insesgado" class="quarto-xref">Teorema 1</a>.

Figura 2: La validación simple dentro de $\mathcal{D}_{\mathrm{train}}$. Los dos primeros bloques constituyen los datos de desarrollo; el tercero es $\mathcal{D}_{\mathrm{test}}$ y permanece aislado hasta el final.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 2.2))

bloques = [
    (0.0, 0.60, "ajuste interno", AZUL, "#e7e9f2",
     "ajustar los\nparámetros"),
    (0.60, 0.20, "validación", NARANJA, "#fff1ea",
     "comparar\nprocedimientos"),
    (0.80, 0.20, "test", TINTA, "#ececeb", "medir una\nsola vez"),
]
for x0, ancho, nombre, color, relleno, uso in bloques:
    ax.add_patch(plt.Rectangle((x0, 0.45), ancho, 0.30, facecolor=relleno,
                               edgecolor=color, lw=1.4))
    ax.text(x0 + ancho / 2, 0.60, nombre, ha="center", va="center",
            fontsize=9.5, color=color)
    ax.text(x0 + ancho / 2, 0.30, uso, ha="center", va="top", fontsize=8.5,
            color=color)

ax.annotate("", xy=(0.80, 0.85), xytext=(0.0, 0.85),
            arrowprops=dict(arrowstyle="|-|", color=GRIS, lw=0.9,
                            mutation_scale=3))
ax.text(0.40, 0.89, r"$\mathcal{D}_{\mathrm{train}}$: desarrollo",
        ha="center", fontsize=8, color=GRIS)
ax.annotate("", xy=(1.0, 0.85), xytext=(0.80, 0.85),
            arrowprops=dict(arrowstyle="|-|", color=TINTA, lw=0.9,
                            mutation_scale=3))
ax.text(0.90, 0.89, r"$\mathcal{D}_{\mathrm{test}}$: una vez",
        ha="center", fontsize=8, color=TINTA)

ax.set(xlim=(-0.03, 1.03), ylim=(0.15, 0.98))
ax.set_axis_off()
plt.tight_layout()

Las proporciones del dibujo son ilustrativas, no una regla. El tamaño absoluto de $\mathcal{D}_{\mathrm{test}}$ determina la precisión de la Tarea 1 por <a href="#prp-precision-test" class="quarto-xref">Proposición 2</a>; el reparto interno de $\mathcal{D}_{\mathrm{train}}$ controla el compromiso entre ajuste y validación que estudiaremos enseguida.

Llegados a este punto, habiendo encontrado la alternativa con menor error en $\mathcal{D}_{\mathrm{val}}$, ¿por qué no entregamos directamente ese modelo ya ajustado? La respuesta está en el tamaño de la muestra: el riesgo esperado de un procedimiento suele decrecer a medida que aumenta el número de observaciones con que se le alimenta. Quedarse con el modelo ajustado sobre $n_{\mathrm{train}}-n_{\mathrm{val}}$ observaciones desperdicia las $n_{\mathrm{val}}$ que se apartaron (y suele resultar en un error pesimista). Por ello la práctica estándar es tomar la alternativa ganadora en validación y reajustarla con todo $\mathcal{D}_{\mathrm{train}}$, antes de abrir el test.

No obstante, al tomar esta decisión, el modelo que hemos evaluado en $\mathcal{D}_{\mathrm{val}}$ (ajustado con $n_{\mathrm{train}}- n_{\mathrm{val}}$ datos) y el modelo que finalmente vamos a entregar (ajustado con $n_{\mathrm{train}}$ datos) no son el mismo modelo. Esto provoca un **cambio conceptual**. Como el modelo evaluado no es el modelo entregado, perdemos la capacidad de estimar el riesgo de modelos concretos durante la fase de selección. Lo que realmente pasamos a evaluar y comparar son procedimientos de aprendizaje.

Un procedimiento $m$ no es una ecuación con parámetros fijos; es la “receta” algorítmica que, al recibir una muestra genérica $\mathcal{D}$ de tamaño $N$, produce un modelo concreto $\hat{f}^{(m)}_{\mathcal{D}}$. Al hacer validación simple, no estamos midiendo si el “Modelo A” es mejor que el “Modelo B”, sino si la “Receta A” produce mejores modelos que la “Receta B” cuando se alimenta con conjuntos de datos de tamaño $n_{\mathrm{train}}- n_{\mathrm{val}}$. El objeto matemático que estamos intentando aproximar al elegir candidatos es, por tanto, el riesgo esperado del procedimiento.

<span class="theorem-title">**Definición 4 (Riesgo esperado de un procedimiento)**</span> Sea $\hat{f}^{(m)}_{\mathcal{D}}$ el modelo concreto que produce el procedimiento $m$ al recibir una muestra $\mathcal{D}$ de tamaño $N$. El **riesgo esperado del procedimiento** $m$ a tamaño $N$ es

$$
\bar{R}_{N}^{(m)}
=\mathbb{E}_{\mathcal{D}\sim(P^\star)^N}\!\left[ R\bigl(\hat{f}^{(m)}_{\mathcal{D}}\bigr) \right] ,
 \qquad(1)$$

donde $R(\hat{f})=\mathbb{E}_{P^\star}\!\left[ \ell(Y,\hat{f}(X)) \right]$ extiende a funciones el riesgo verdadero de **?@def-riesgo-verdadero**.

Esta esperanza promedia el riesgo verdadero de todos los modelos que el procedimiento $m$ produciría al recibir distintas muestras independientes de tamaño $N$ extraídas de la población. Cuando el procedimiento esté claro por el contexto escribiremos solo $\bar{R}_{N}$, dejando el tamaño de muestra a la vista, que es lo que cambia en lo que sigue. En el caso de la validación simple, nuestra evaluación sobre $\mathcal{D}_{\mathrm{val}}$ estima $\bar{R}_{n_{\mathrm{train}}- n_{\mathrm{val}}}^{(m)}$, y es insesgada al promediar sobre las dos fuentes de azar que intervienen: qué observaciones caen en $\mathcal{D}_{\mathrm{ajuste}}$ (que fijan el modelo) y cuáles caen en $\mathcal{D}_{\mathrm{val}}$, (que lo miden). El ejercicio siguiente lo demuestra. Nuestra estrategia subyacente consiste en elegir el procedimiento que tiene el menor riesgo esperado a tamaño $n_{\mathrm{train}}- n_{\mathrm{val}}$, asumiendo que esa ventaja sistemática se mantendrá cuando le entreguemos los $n_{\mathrm{train}}$ datos completos para forjar el modelo final.

<span class="theorem-title">**Ejercicio 3 (La validación simple es insesgada)**</span> Sea $\mathcal{D}_{\mathrm{train}}$ una muestra de $n_{\mathrm{train}}$ extracciones independientes de $P^\star$, repartida **al azar y sin mirar los datos** en $\mathcal{D}_{\mathrm{ajuste}}$, con $n_{\mathrm{train}}-n_{\mathrm{val}}$ observaciones, y $\mathcal{D}_{\mathrm{val}}$, con $n_{\mathrm{val}}$. El procedimiento $m$ recibe $\mathcal{D}_{\mathrm{ajuste}}$ y devuelve $\hat{f}^{(m)}_{\mathcal{D}_{\mathrm{ajuste}}}$, que se mide sobre $\mathcal{D}_{\mathrm{val}}$.

1.  **Razona** que, fijado $\mathcal{D}_{\mathrm{ajuste}}$, el modelo es un objeto fijo y las observaciones de $\mathcal{D}_{\mathrm{val}}$ le son ajenas, de modo que <a href="#thm-test-insesgado" class="quarto-xref">Teorema 1</a> se aplica tomando $\mathcal{D}_{\mathrm{val}}$ como muestra de evaluación, y **escribe** la esperanza condicional que resulta.
2.  **Toma** esperanzas sobre $\mathcal{D}_{\mathrm{ajuste}}$ y **concluye**, con <a href="#def-riesgo-procedimiento" class="quarto-xref">Definición 4</a>, que $\mathbb{E}_{\mathcal{D}_{\mathrm{train}}}\!\left[ \hat{R}_{\mathcal{D}_{\mathrm{val}}}(\hat{f}^{(m)}_{\mathcal{D}_{\mathrm{ajuste}}}) \right]
    =\bar{R}_{n_{\mathrm{train}}-n_{\mathrm{val}}}^{(m)}$.
3.  **Di** qué paso deja de valer si el reparto se hace mirando los datos, por ejemplo enviando a $\mathcal{D}_{\mathrm{val}}$ las observaciones peor predichas por un ajuste previo.

Si la distribución $P^\star$ fuera conocida, podríamos aproximar la esperanza de <a href="#eq-riesgo-procedimiento" class="quarto-xref">Ecuación 1</a> para cada candidato. Bastaría con generar muestras $\mathcal{D}_1,\mathcal{D}_2,\ldots$ de tamaño $N$, aplicar el procedimiento para producir un modelo nuevo en cada una y evaluar su riesgo verdadero sobre la población: por la ley de los grandes números, la media

$$
\frac{1}{B}\sum_{b=1}^{B}R\bigl(\hat{f}^{(m)}_{\mathcal{D}_b}\bigr)
$$

converge a $\bar{R}_{N}^{(m)}$ cuando $B\to\infty$. La selección ideal sería entonces el procedimiento $m^\star$ de menor $\bar{R}_{N}^{(m)}$.

Como en la práctica este experimento es imposible (no tenemos infinitas muestras ni $P^\star$), la validación simple es nuestra herramienta para imitarlo empíricamente utilizando un único conjunto de datos. Con todo lo anterior, la distinción fundamental para enfrentarnos al resto del capítulo queda fijada: $R(\hat{\boldsymbol{w}})$ es el riesgo verdadero de un modelo concreto, y se estima abriendo $\mathcal{D}_{\mathrm{test}}$ una sola vez al final del proceso. $\bar{R}_{N}^{(m)}$ es el riesgo esperado de un procedimiento, y se estima internamente mediante particiones dentro de $\mathcal{D}_{\mathrm{train}}$ para elegir entre alternativas.

<span class="theorem-title">**Ejercicio 4 (Selección sobre ruido puro)**</span> Repite el experimento de <a href="#fig-sesgo-seleccion" class="quarto-xref">Figura 1</a> quitando la señal, es decir, generando `y` como ruido normal sin ninguna dependencia de `x`, y con candidatos que usen **solo** su variable inútil.

1.  Di cuál es el riesgo verdadero de todos los candidatos antes de ejecutar nada.
2.  Ejecútalo con $M=100$, promediando sobre 200 repeticiones, y compara el riesgo del modelo elegido sobre el conjunto con el que se eligió con su riesgo de test.
3.  Explica qué tendría que mirar un analista para darse cuenta de que su modelo no ha aprendido nada.

### K-fold cross validation

La validación simple busca aproximar $\bar{R}_{n_{\mathrm{train}}}$, el riesgo esperado del procedimiento cuando recibe todo $\mathcal{D}_{\mathrm{train}}$, que es el tamaño con el que se ajustará el modelo entregado. Este procedimiento paga dos costes simultáneos.

- **Sesgo pesimista.** Si $\mathcal{D}_{\mathrm{val}}$ contiene el 20 % de $\mathcal{D}_{\mathrm{train}}$, cada procedimiento se aplica a solo $0.8n_{\mathrm{train}}$ observaciones. La validación estima entonces $\bar{R}_{0.8n_{\mathrm{train}}}$, no $\bar{R}_{n_{\mathrm{train}}}$. Como el riesgo esperado suele decrecer con el tamaño muestral, se sobrestima el riesgo esperado correspondiente al uso de todo $\mathcal{D}_{\mathrm{train}}$.
- **Varianza alta.** La comparación usa solo $n_{\mathrm{val}}$ pérdidas, y por <a href="#prp-precision-test" class="quarto-xref">Proposición 2</a> su error típico es de orden $1/\sqrt{n_{\mathrm{val}}}$, que puede superar las diferencias entre candidatos.

La **validación cruzada de $K$ bloques** (*K-fold cross validation*) es una alternativa poderosa a la validación simple.

<span class="theorem-title">**Definición 5 (K-fold cross validation)**</span> Se particiona el conjunto de entrenamiento $\mathcal{D}_{\mathrm{train}}$ al azar en $K$ bloques disjuntos $\mathcal{F}_{1},\ldots,\mathcal{F}_{K}$ de tamaño aproximadamente igual. Para cada bloque $k \in \{1,\ldots,K\}$:

1.  Se entrena el procedimiento utilizando todos los datos excepto los contenidos en $\mathcal{F}_{k}$.

2.  Se evalúa el modelo resultante sobre el bloque retenido $\mathcal{F}_{k}$ para obtener su riesgo, denotado como $\hat{R}_{\mathcal{F}_{k}}$.

3.  La estimación final del riesgo por K-fold cross validation ($\mathrm{CV}$) se obtiene promediando estas $K$ evaluaciones independientes:

$$
\mathrm{CV}=\frac{1}{K}\sum_{k=1}^{K}\hat{R}_{\mathcal{F}_{k}} .
 \qquad(2)$$

Con este diseño, cada observación de $\mathcal{D}_{\mathrm{train}}$ evalúa exactamente una vez y ajusta $K-1$ veces. Así, la estimación $\mathrm{CV}$ logra aprovechar la totalidad de la muestra disponible sin violar jamás la regla fundamental de evaluar sobre datos no vistos.

El reparto en bloques debe ser estrictamente aleatorio para garantizar que cada partición represente la misma distribución poblacional.

Figura 3: K-fold cross validation con $K=5$. Cada fila aplica el procedimiento a cuatro bloques y evalúa el modelo resultante con el que queda. La estimación es la media de las cinco evaluaciones.

In [ ]:
K = 5
fig, ax = plt.subplots(figsize=(7.2, 2.9))

for fila in range(K):
    for bloque in range(K):
        es_val = bloque == fila
        ax.add_patch(plt.Rectangle(
            (bloque, -fila), 0.94, 0.8,
            facecolor="#fff1ea" if es_val else "#e7e9f2",
            edgecolor=NARANJA if es_val else AZUL, lw=1.1,
        ))
    ax.text(-0.25, -fila + 0.4, f"$k={fila + 1}$", ha="right", va="center",
            fontsize=9, color=GRIS)
    ax.text(K + 0.15, -fila + 0.4,
            r"$\hat R_{\mathcal{F}_%d}$" % (fila + 1),
            ha="left", va="center", fontsize=9, color=NARANJA)

ax.text(K / 2, 1.15, "los cinco bloques en que se reparte train",
        ha="center", fontsize=9, color=GRIS)
ax.text(2.5, -K + 0.65,
        "azul: se ajusta con ellos      naranja: se evalúa con él",
        ha="center", va="top", fontsize=8.5, color=GRIS)

ax.set(xlim=(-1.4, K + 1.5), ylim=(-K + 0.15, 1.45))
ax.set_axis_off()
plt.tight_layout()

La cantidad de <a href="#eq-cv" class="quarto-xref">Ecuación 2</a> no estima el riesgo de un modelo concreto: promedia $K$ modelos distintos. Cada uno procede de aproximadamente $n_{\mathrm{train}}(K-1)/K$ observaciones, de modo que la cantidad estimada es $\bar{R}_{n_{\mathrm{train}}(K-1)/K}$, no $\bar{R}_{n_{\mathrm{train}}}$. El ejercicio siguiente lo demuestra, con el mismo argumento de <a href="#exr-validacion-insesgada" class="quarto-xref">Ejercicio 3</a>.

<span class="theorem-title">**Ejercicio 5 (Qué estima exactamente la validación cruzada)**</span> Con las hipótesis de <a href="#exr-validacion-insesgada" class="quarto-xref">Ejercicio 3</a>, reparte $\mathcal{D}_{\mathrm{train}}$ al azar en $K$ bloques del mismo tamaño y llama $n_a=n_{\mathrm{train}}(K-1)/K$ al número de observaciones de cada conjunto de ajuste.

1.  **Aplica** el apartado b de <a href="#exr-validacion-insesgada" class="quarto-xref">Ejercicio 3</a> a un bloque cualquiera $\mathcal{F}_{k}$ y **deduce** que $\mathbb{E}_{\mathcal{D}_{\mathrm{train}}}\!\left[ \hat{R}_{\mathcal{F}_{k}} \right]=\bar{R}_{n_a}^{(m)}$ para todo $k$.
2.  **Concluye** que $\mathbb{E}_{\mathcal{D}_{\mathrm{train}}}\!\left[ \mathrm{CV} \right]=\bar{R}_{n_a}^{(m)}$ y **di** qué propiedad de la esperanza usas. **Observa** que no hace falta que los $K$ riesgos sean independientes.
3.  Con $n_{\mathrm{train}}=67$ y $K=5$ los bloques no pueden ser iguales, y los ajustes usan 53 o 54 observaciones. **Di** de qué es media entonces $\mathbb{E}_{\mathcal{D}_{\mathrm{train}}}\!\left[ \mathrm{CV} \right]$.
4.  **Explica** por qué nada de esto dice que $\mathrm{CV}$ sea insesgada para $\bar{R}_{n_{\mathrm{train}}}$, ni para el riesgo verdadero del modelo concreto que se acabará entregando.

Frente a la validación simple, K-fold mitiga los dos costes:

- reduce el sesgo pesimista porque $n_{\mathrm{train}}(K-1)/K$ está más cerca de $n_{\mathrm{train}}$;
- la estimación tiene menos varianza porque todas las observaciones de $\mathcal{D}_{\mathrm{train}}$ contribuyen a la evaluación.

**La corrección no es exacta.** Elegir mediante $\bar{R}_{n_{\mathrm{train}}(K-1)/K}$ supone que el orden de los procedimientos no cambia al aplicarlos a $n_{\mathrm{train}}$ observaciones. Además, los $K$ riesgos no son independientes: dos ajustes comparten una fracción $(K-2)/(K-1)$ de sus datos. Por eso su promedio reduce la varianza menos que $K$ mediciones independientes. Estas dos limitaciones explican la convención $K=5$ o $K=10$: valores mayores reducen algo el sesgo, pero elevan el coste y no garantizan una menor varianza. Con muestras muy grandes, una validación simple puede ser preferible porque tanto la pérdida de datos de ajuste como el error típico sobre $\mathcal{D}_{\mathrm{val}}$ se vuelven despreciables. En muestras pequeñas conviene fijar la semilla y examinar la sensibilidad al reparto, como propone <a href="#exr-cv-bloques" class="quarto-xref">Ejercicio 6</a>.

Apliquemos K-fold cross validation a los datos de próstata. Los procedimientos candidatos usan un número creciente de variables, ordenadas por su correlación absoluta con la respuesta.

In [ ]:
# TODO: completar en clase

El orden lo encabeza `lcavol`, con $+0.733$, muy por delante del resto. Las posiciones tercera y cuarta se juegan por tres milésimas, `lcp` con $+0.489$ frente a `lweight` con $+0.485$, y esa diferencia mínima decide qué variables entran en el candidato que acabaremos entregando.

In [ ]:
# TODO: completar en clase

In [ ]:
# TODO: completar en clase

La curva tiene un mínimo en el interior: baja de $1.4655$ para el procedimiento nulo hasta $0.5611$ para el de cinco variables, y vuelve a subir hasta $0.6144$ para el de ocho.

Figura 4: La estimación por K-fold cross validation frente al número de variables. Los puntos grises son los cinco riesgos que se promedian en cada caso. Su dispersión es grande: en el mínimo van de $0.22$ a $1.09$, de modo que el número promediado lleva bastante incertidumbre.

In [ ]:
fig, ax = plt.subplots(figsize=(6.4, 4.0))

ms = [fila[0] for fila in tabla_cv]
for m, cv, bloques in tabla_cv:
    ax.scatter([m] * len(bloques), bloques, color=GRIS, s=14, alpha=0.7,
               zorder=2)
ax.plot(ms, [fila[1] for fila in tabla_cv], marker="o", ms=5, color=NARANJA,
        lw=1.6, zorder=3, label=r"$\mathrm{CV}(5)$")
ax.scatter([m_campeon], [dict((f[0], f[1]) for f in tabla_cv)[m_campeon]],
           marker="*", s=200, facecolor="white", edgecolor="black", lw=0.9,
           zorder=4, label="mínimo")
ax.set(xlabel="número de variables predictoras",
       ylabel="riesgo cuadrático medio",
       title="K-fold cross validation sobre las 67 observaciones de train")
ax.legend()
plt.tight_layout()

<span class="theorem-title">**Ejercicio 6 (Cuántos bloques)**</span>  

1.  Con $K=n_{\mathrm{train}}$ cada bloque tiene una sola observación, y el procedimiento se llama *leave-one-out*. Di cuántos ajustes hacen falta y por qué eso lo vuelve caro.
2.  Con $K$ grande, el procedimiento fija cada modelo usando casi todas las observaciones. Razona en qué dirección cambia entonces el sesgo de <a href="#eq-cv" class="quarto-xref">Ecuación 2</a> como estimación de $\bar{R}_{n_{\mathrm{train}}}$.
3.  Ejecuta `cv_riesgo` para $m=5$ con `n_bloques` igual a 3, 5 y 10, y con tres semillas distintas en cada caso. Comenta cuánta variación hay entre semillas frente a cuánta hay entre números de bloques.

### Comparar candidatos bloque a bloque

Si comparamos directamente las estimaciones globales $\mathrm{CV}$ de la tabla, su varianza nos llevaría a concluir que los candidatos son indistinguibles. Por ejemplo, la diferencia entre usar cinco o seis variables es de apenas $0.023$, frente a un error típico individual que ronda el $0.150$.Esta comparación ingenua ignora el diseño emparejado del experimento. Como ambos procedimientos se evalúan sobre los mismos bloques, un bloque particularmente difícil inflará el error de ambos modelos a la vez. Al restar sus riesgos bloque a bloque, esa variación compartida se cancela. Por este motivo, el error típico de la diferencia emparejada es usualmente mucho menor que el error típico de las medidas individuales, permitiéndonos detectar diferencias reales con mayor precisión.

<span class="theorem-title">**Definición 6 (Diferencia emparejada)**</span> Sean dos procedimientos $A$ y $B$ evaluados sobre los mismos $K$ bloques de validación cruzada.

1.  Diferencias por bloque: para cada bloque $k \in \{1, \dots, K\}$, se calcula:

$$
d_k = \hat{R}_{A,\mathcal{F}_{k}} - \hat{R}_{B,\mathcal{F}_{k}}$$

1.  Diferencia media ($\bar{d}$): es el promedio de las diferencias:

$$
\bar{d} = \frac{1}{K} \sum_{k=1}^K d_k = \mathrm{CV}_A - \mathrm{CV}_B
$$

1.  Error típico emparejado ($\text{EE}_{\text{dif}}$): mide la incertidumbre de $\bar{d}$ a partir de la desviación típica muestral de las diferencias ($s_d$):

$$
s_d = \sqrt{\frac{1}{K-1} \sum_{k=1}^K (d_k - \bar{d})^2}, \qquad \text{EE}_{\text{dif}} = \frac{s_d}{\sqrt{K}}
$$

Con los datos de la tabla, la diferencia media entre seis y cinco variables es $\bar d=+0.0228$. Al emparejarlos, su error típico se desploma a $0.0129$ (más de diez veces menor que los $0.150$ individuales).

``` python
m_siguiente = m_campeon + 1
a, b = bloques_de[m_campeon], bloques_de[m_siguiente]
d = b - a
ee = d.std(ddof=1) / np.sqrt(len(d))
posicion = np.arange(1, len(d) + 1)

fig, ax = plt.subplots(1, 2, figsize=(8.4, 3.0))

ax[0].plot(posicion, a, marker="o", ms=5, color=AZUL,
           label=f"{m_campeon} variables")
ax[0].plot(posicion, b, marker="s", ms=5, color=NARANJA,
           label=f"{m_siguiente} variables")
ax[0].set(xlabel="bloque", ylabel="riesgo del bloque",
          title="los riesgos de cada candidato")
ax[0].set_xticks(posicion)
ax[0].legend(fontsize=8.5)

ax[1].fill_between([0.5, len(d) + 0.5], d.mean() - ee, d.mean() + ee,
                   color=NARANJA, alpha=0.18,
                   label="media y un error típico")
ax[1].axhline(d.mean(), color=NARANJA, lw=1.5)
ax[1].axhline(0.0, color=GRIS, lw=1, linestyle=":")
ax[1].plot(posicion, d, marker="o", ms=5, ls="none", color=TINTA,
           label="diferencia del bloque")
ax[1].set(xlabel="bloque", ylabel="diferencia",
          xlim=(0.5, len(d) + 0.5), title="sus diferencias")
ax[1].set_xticks(posicion)
ax[1].legend(fontsize=8.5, loc="lower right")

plt.tight_layout()
```

Figura 5: Evaluación de los mismos cinco bloques con dos candidatos. Izquierda: el tercer bloque es difícil para ambos y el quinto es fácil para ambos (alta varianza compartida). Derecha: al restar bloque a bloque ($d_k$), esa varianza desaparece y la diferencia se mide con mucha más precisión (en una escala más estrecha).

**El error típico emparejado tampoco es exacto.** La fórmula de $\text{EE}_{\text{dif}}$ supone que las $K$ diferencias son independientes, y con validación cruzada no lo son, por el solapamiento entre bloques que ya vimos en <a href="#sec-kfold" class="quarto-xref">Sección 4.2</a>. Por tanto, $\text{EE}_{\text{dif}}$ es una escala operativa de incertidumbre, no el error típico exacto de $\bar d$. La regla que sigue es una heurística de selección, no un contraste de hipótesis.

#### Cómo elegir entre dos candidatos

Si la competición fuera exclusivamente entre dos candidatos, la decisión sería directa con estos dos valores. Basta dividir la diferencia emparejada entre su error típico para ver cuántas veces supera la diferencia a la escala del ruido con que se ha medido:

- Si el cociente pasa de $2$, la diferencia es difícil de atribuir al ruido de la medición y se afirma que el candidato de menor riesgo es mejor.

- Si el cociente no llega a $2$, la diferencia no se distingue del ruido. En ese empate se entrega el candidato más sencillo, por parsimonia.

Con dos candidatos ninguno es la referencia natural, así que lo que se compara con el umbral es el valor absoluto, $\left\lvert \bar d \right\rvert/\text{EE}_{\text{dif}}$. Nótese que el umbral $2$ es convencional.

#### Cómo elegir entre múltiples candidatos

Si en lugar de dos tenemos múltiples candidatos, compararlos todos contra todos es estadísticamente peligroso: cuantas más comparaciones cruzadas hagamos, más probable será encontrar una supuesta ventaja que sea puro ruido (<a href="#prp-minimo-sesgado" class="quarto-xref">Proposición 3</a>).

Para evitar esta multiplicidad y penalizar la complejidad innecesaria, un estándar práctico es emplear una **heurística** conocida como la regla de un error típico.

<span class="theorem-title">**Definición 7 (Regla de un error típico)**</span> Ordenados los candidatos del más simple al más complejo, y tras estimar el riesgo de cada uno por validación cruzada sobre los mismos $K$ bloques:

1.  Identificar al campeón. Localizar el candidato que haya obtenido el menor riesgo medio global, $\mathrm{CV}_{\min}$.

2.  Emparejar contra el campeón. Para cada candidato más simple que el campeón, calcular la diferencia emparejada $\bar d$ y su error típico $\text{EE}_{\text{dif}}$ frente a él, según <a href="#def-diferencia-emparejada" class="quarto-xref">Definición 6</a>.

3.  Decidir. Elegir el procedimiento más simple cuya desventaja frente al campeón sea, como máximo, de un error típico ($\bar{d} / \text{EE}_{\text{dif}} \le 1$). Si ninguno entra dentro de este margen, se elige al campeón.

Esta regla reconoce que el riesgo del campeón está favorecido por el azar y aplica el principio de parsimonia (la navaja de Ockham): si la ventaja numérica del ganador absoluto frente a una alternativa más sencilla cabe dentro del propio ruido de la medición emparejada, sencillamente no hay evidencia operativa para justificar esa complejidad adicional.

Aplicado al caso de la próstata:

In [ ]:
# TODO: completar en clase

La última columna toma la decisión:

- 3 variables, el procedimiento elegido: está a 0.9 errores típicos, por debajo de 1. La regla lo prefiere porque ahorra dos variables frente al campeón y su desventaja queda dentro del ruido de la medición.

- El procedimiento nulo queda lejos del campeón: su desventaja equivale a 4.1 errores típicos.

- 1 y 2 variables: sus cocientes pasan de 1, aunque por poco. La columna los redondea a 1.0 y ambos valen 1.04, así que la regla los descarta justo en el límite: con `semilla=0` en el reparto en bloques entran los dos y la regla entrega el candidato de una sola variable. Es un recordatorio de que la regla es una heurística, no un veredicto, y de que el reparto en bloques es una fuente de variabilidad que conviene fijar e informar.

- 4 variables: aunque su diferencia absoluta es la más pequeña, $+0.031$, su cociente es 1.7. Los modelos que produce en cada bloque son casi idénticos a los del campeón, así que ambos procedimientos fallan en los mismos bloques. El error típico emparejado es solo $0.019$ y la pequeña desventaja se mide con mucha precisión.

## Métricas de regresión

Hasta aquí hemos medido la calidad del modelo con el error cuadrático medio, que es el que salió de la verosimilitud gaussiana en el capítulo 2. Para informar de un resultado, habitualmente se usan además otras tres cantidades, todas construidas sobre los mismos residuos.

<span class="theorem-title">**Definición 8 (Métricas de regresión)**</span> Sean $\hat{y}_i$ las predicciones sobre una muestra de tamaño $n$ y $r_i=y_i-\hat{y}_i$ los residuos. Se definen

$$
\mathrm{MSE}=\frac{1}{n}\sum_{i=1}^{n}r_i^2,
\qquad
\mathrm{RMSE}=\sqrt{\mathrm{MSE}},
\qquad
\mathrm{MAE}=\frac{1}{n}\sum_{i=1}^{n}\left\lvert r_i \right\rvert,
$$

y el **coeficiente de determinación**

$$
R^2=1-\frac{\mathrm{SSE}}{\mathrm{SST}},
\qquad
\mathrm{SSE}=\sum_{i=1}^{n}r_i^2,
\qquad
\mathrm{SST}=\sum_{i=1}^{n}(y_i-\bar{y})^2 .
 \qquad(3)$$

Las cuatro cantidades dicen cosas distintas. El $\mathrm{MSE}$ está en unidades de la respuesta **al cuadrado**, de modo que su valor no se puede comparar con el rango de $y$. El $\mathrm{RMSE}$ y el $\mathrm{MAE}$ están en las unidades de la respuesta y sí se pueden comparar con ese rango. Los dos resumen el tamaño típico de un residuo, pero el $\mathrm{RMSE}$ eleva al cuadrado antes de promediar, así que una sola observación con residuo grande lo mueve mucho más que al $\mathrm{MAE}$. Y el $R^2$ no tiene unidades. Por <a href="#eq-r2" class="quarto-xref">Ecuación 3</a> y <a href="#lem-media-optima" class="quarto-xref">Lema 1</a>, mide la fracción del error cuadrático de la mejor constante **sobre la muestra evaluada** que elimina el modelo. En $\mathcal{D}_{\mathrm{train}}$ esa constante coincide con el modelo nulo; en $\mathcal{D}_{\mathrm{test}}$ no tiene por qué coincidir, porque el modelo nulo se fijó sin usar las respuestas de test. Un ajuste perfecto da $R^2=1$.

Las dos métricas en unidades de la respuesta están además ordenadas, y el ejercicio siguiente lo demuestra.

<span class="theorem-title">**Ejercicio 7 (El RMSE nunca es menor que el MAE)**</span> Sean $a_i=\left\lvert r_i \right\rvert$ los residuos en valor absoluto y sea $\bar a$ su media, que es el $\mathrm{MAE}$.

1.  Desarrolla la varianza muestral de los $a_i$ y **comprueba** que $\frac{1}{n}\sum_i(a_i-\bar a)^2=\mathrm{MSE}-\mathrm{MAE}^2$, usando que $a_i^2=r_i^2$.
2.  **Deduce** que $\mathrm{RMSE}\geq\mathrm{MAE}$ para cualquier vector de residuos, y **di** en qué caso se da la igualdad.
3.  **Explica** qué indica, sobre el reparto de los residuos, que el $\mathrm{RMSE}$ sea mucho mayor que el $\mathrm{MAE}$.

Calculemos las cuatro métricas sobre $\mathcal{D}_{\mathrm{train}}$ para ilustrar sus escalas. Estos valores no son estimaciones finales de generalización.

In [ ]:
# TODO: completar en clase

En entrenamiento, el modelo lineal tiene $\mathrm{RMSE}=0.6627$ y $\mathrm{MAE}=0.4986$, de modo que la desigualdad de <a href="#exr-rmse-mae" class="quarto-xref">Ejercicio 7</a> se cumple con holgura. Además, el $R^2$ del modelo nulo es cero por construcción.

**El $R^2$ puede ser negativo fuera de entrenamiento.** Por <a href="#eq-r2" class="quarto-xref">Ecuación 3</a>, $R^2<0$ significa $\mathrm{SSE}>\mathrm{SST}$, es decir, que el modelo predice peor que la media de la muestra sobre la que se mide. En train eso no puede pasar con una arquitectura lineal que incluya el intercepto, porque la familia contiene las funciones constantes y el procedimiento no devuelve una ecuación peor. En test sí puede ocurrir: los parámetros se fijaron con $\mathcal{D}_{\mathrm{train}}$, mientras que $\mathrm{SST}$ usa la media de las respuestas de $\mathcal{D}_{\mathrm{test}}$.

<span class="theorem-title">**Ejercicio 8 (Leer las cuatro métricas)**</span>  

1.  **Comprueba**, con los números que imprime el bloque anterior, que el $R^2$ del modelo nulo medido en train es exactamente cero, y **explica** por qué, usando <a href="#lem-media-optima" class="quarto-xref">Lema 1</a> y <a href="#eq-r2" class="quarto-xref">Ecuación 3</a>.
2.  **Demuestra** que, sobre una muestra fija, ordenar varios modelos por $R^2$ y ordenarlos por $\mathrm{MSE}$ da el mismo orden. **Di** qué deja de valer si los dos modelos se miden sobre muestras distintas.

## Caso práctico: abrir el test una sola vez

Hasta este punto, hemos usado exclusivamente las 67 observaciones de $\mathcal{D}_{\mathrm{train}}$. La validación cruzada y la regla de un error típico seleccionaron el procedimiento de tres variables. Ahora aplicamos este procedimiento a $\mathcal{D}_{\mathrm{train}}$ completo para fijar el modelo definitivo. Solo entonces abrimos las 30 observaciones de $\mathcal{D}_{\mathrm{test}}$.

In [ ]:
# TODO: completar en clase

El modelo entregado usa lcavol, svi y lcp. Los tres riesgos impresos no coinciden porque miden objetos conceptualmente distintos:

1.  Riesgo en train ($0.6159$): Es una estimación engañosamente optimista (<a href="#prp-train-optimista" class="quarto-xref">Proposición 1</a>) porque evalúa el modelo sobre los mismos datos usados para ajustarlo.

2.  $\mathrm{CV}$ ($0.6558$): Estima el riesgo del procedimiento cuando este se entrena con muestras de 53 o 54 observaciones.

3.  Riesgo en test ($0.4428$): Es la estimación insesgada del riesgo del modelo concreto que entregamos (ajustado con 67 observaciones). Que aquí resulte el número más bajo es producto de la variabilidad del azar en la partición, ya que se calcula sobre solo 30 datos y arrastra un error típico considerable.

En esta ronda final medimos también las dos referencias fijadas de antemano: el modelo nulo y el modelo lineal de ocho variables.

In [ ]:
# TODO: completar en clase

El modelo nulo da $1.0567$, el de ocho variables $0.5213$ y el modelo final de tres variables $0.4428$. Frente al de ocho variables, la diferencia emparejada es $-0.0784\pm0.1061$: la mejora no alcanza un error típico y la razón para preferir el modelo final es que usa cinco variables menos.

Frente al modelo nulo, la diferencia es $-0.6139\pm0.3092$, un cociente de $1.99$. La mejora queda justo en el umbral convencional de dos errores típicos; un test mayor la mediría con más precisión.

Figura 6: El modelo entregado, el de tres variables, medido después de cerrar la selección. Su riesgo es $0.6159$ en train y $0.4428$ en test.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(8, 4.0))

for panel, (Xp, yp, color, titulo) in enumerate([
    (X_train, y_train, AZUL, "train, 67 observaciones"),
    (X_test, y_test, TINTA, "test, 30 observaciones"),
]):
    pred = modelo_final.predict(Xp[:, cols_elegidas])
    limites = [y.min().item() - 0.3, y.max().item() + 0.3]
    ax[panel].plot(limites, limites, color=GRIS, lw=1, linestyle="--")
    ax[panel].scatter(yp.numpy(), pred.numpy(), color=color, **kw_puntos,
                      zorder=3)
    ax[panel].set(xlabel="lpsa observado", ylabel="lpsa predicho",
                  xlim=limites, ylim=limites,
                  title=f"{titulo}\nriesgo {mse(pred, yp):.4f}",
                  aspect="equal")

plt.tight_layout()

## El protocolo reproducible

Lo visto en el capítulo, se resume en el siguiente protocolo.

1.  Separar test. Extraer $\mathcal{D}_{\mathrm{test}}$ al azar y aislarlo por completo hasta que no quede ninguna decisión algorítmica pendiente. Esta es la condición que exige la hipótesis de <a href="#thm-test-insesgado" class="quarto-xref">Teorema 1</a>.

2.  Fijar la referencia. Ajustar el procedimiento nulo (<a href="#def-modelo-nulo" class="quarto-xref">Definición 1</a>) utilizando exclusivamente $\mathcal{D}_{\mathrm{train}}$ y establecer de antemano las métricas que se reportarán al final.

3.  Elegir el procedimiento. Comparar las alternativas mediante K-fold cross validation dentro de $\mathcal{D}_{\mathrm{train}}$ y decidir aplicando la regla de un error típico con diferencias emparejadas bloque a bloque. En este paso comparamos el riesgo esperado de cada procedimiento (<a href="#eq-riesgo-procedimiento" class="quarto-xref">Ecuación 1</a>). Cualquier cálculo que guíe la decisión, incluido el ordenamiento previo de las variables, debe ejecutarse estrictamente sobre $\mathcal{D}_{\mathrm{train}}$ para evitar el sesgo de selección (<a href="#prp-minimo-sesgado" class="quarto-xref">Proposición 3</a>).

4.  Reajustar. Aplicar el procedimiento ganador a la totalidad de $\mathcal{D}_{\mathrm{train}}$ para fijar los parámetros del modelo definitivo. Como lo que hemos seleccionado es una “receta” algorítmica y no un modelo terminado, es obligatorio volver a ejecutarla aprovechando toda la muestra de desarrollo, que es el tamaño de datos al que la validación cruzada aspiraba a extrapolar.

5.  Medir una sola vez. Abrir $\mathcal{D}_{\mathrm{test}}$, evaluar el modelo definitivo y reportar su riesgo junto con el error típico y el valor de la referencia nula. Gracias al aislamiento estricto, esta estimación final es honesta (<a href="#thm-test-insesgado" class="quarto-xref">Teorema 1</a>).

Con este protocolo ya se puede informar de un resultado sin engañarse. Queda un supuesto grande sin examinar: hemos trabajado con una tabla de números limpia, donde lo único que se aprendía de los datos eran los coeficientes. En una tabla real hay que imputar ausencias, escalar columnas y codificar categorías, y todas esas transformaciones se aprenden también de los datos. Si se aprenden antes de separar, el test deja de ser test y ninguno de los resultados de este capítulo se aplica. Ese es el asunto del capítulo siguiente.

## Ejercicios

<span class="theorem-title">**Ejercicio 9 (El protocolo con otra partición)**</span> Repite el protocolo completo de <a href="#sec-protocolo" class="quarto-xref">Sección 7</a>, pero sustituyendo la partición del libro por una al azar con `sklearn.model_selection.train_test_split` y `random_state=0`, manteniendo 30 observaciones de test.

1.  Anota el $m$ elegido, la estimación por K-fold cross validation y el riesgo en test.
2.  Repite con cinco semillas distintas y construye una tabla con los cinco resultados.
3.  Compara la variación entre semillas con el error típico de <a href="#prp-precision-test" class="quarto-xref">Proposición 2</a>, y di qué parte de esa variación viene de estimar con pocas observaciones y qué parte de que el modelo elegido cambie.

<span class="theorem-title">**Ejercicio 10 (Qué pasa si el test se mira dos veces)**</span> Simula el ciclo que <a href="#thm-test-insesgado" class="quarto-xref">Teorema 1</a> prohíbe. Con los datos de <a href="#exr-sesgo-sin-senal" class="quarto-xref">Ejercicio 4</a>, aplica 20 procedimientos candidatos, elige el mejor **por su riesgo de test** y anota ese riesgo. Después mide ese mismo modelo elegido sobre un conjunto nuevo e independiente.

1.  Compara los dos números y explica el resultado con <a href="#prp-minimo-sesgado" class="quarto-xref">Proposición 3</a>.
2.  Explica en qué se ha convertido el conjunto de test en cuanto se ha usado para elegir.
3.  Propón qué habría que haber hecho para poder informar de un número honesto.

<span class="theorem-title">**Ejercicio 11 (Una observación que domina el error)**</span> Sobre las 30 observaciones de test, calcula las cuatro métricas de <a href="#def-metricas" class="quarto-xref">Definición 8</a>. Después localiza la observación con el residuo más grande en valor absoluto, quítala y recalcula las cuatro.

1.  Con <a href="#exr-rmse-mae" class="quarto-xref">Ejercicio 7</a> a la vista, di cuál de las cuatro métricas cambia proporcionalmente más y por qué.
2.  Explica qué métrica informarías si el coste real de un error creciera de forma proporcional al error, y cuál si creciera más deprisa.

<span class="theorem-title">**Ejercicio 12 (K-fold cross validation como estimador)**</span> Con datos simulados, donde se puede calcular el riesgo verdadero sobre una muestra enorme, compara la estimación por K-fold cross validation con la verdad.

1.  Genera 60 observaciones de train con diez variables predictoras normales independientes, de las cuales solo la primera entra en la señal, $f(\mathbf{x})=1+2x_1$, con $\sigma=1$. Aplica mínimos cuadrados a la arquitectura de once columnas, incluida la de unos, para obtener un modelo concreto.
2.  Calcula $\mathrm{CV}$ con cinco bloques sobre las 60, y el riesgo verdadero del modelo ajustado con las 60 usando 20000 observaciones nuevas.
3.  Repite el experimento 200 veces y compara la media de $\mathrm{CV}$ con la media del riesgo verdadero. Comprueba el signo de la diferencia y relaciónalo con lo dicho tras <a href="#def-validacion-cruzada" class="quarto-xref">Definición 5</a>.
4.  Repite con una sola variable predictora en lugar de diez. Comprueba que con ese modelo 200 repeticiones no bastan para determinar el signo, comparando la diferencia con su error típico, y explica qué tiene que ver el número de parámetros con el tamaño del efecto.

<span class="theorem-title">**Ejercicio 13 (La entrega alternativa)**</span> El caso práctico aplica el procedimiento que elige la regla de un error típico. El mínimo de la curva correspondía a otro procedimiento.

1.  Aplica el procedimiento campeón, el de cinco variables, a las 67 observaciones de train y compara sus tres números con los del modelo entregado.
2.  Calcula la diferencia emparejada entre los dos modelos sobre las 30 observaciones de test, con su error típico, y di si el test distingue uno del otro.
3.  Frente a las ocho variables el emparejamiento reduce el error típico a la mitad, mientras que en <a href="#sec-comparar" class="quarto-xref">Sección 4.3</a> lo reducía mucho más entre candidatos casi iguales. Explica de qué depende esa ganancia.